## Task 1: Data Profiling and Missing Values

In [1]:
# ==============================
# TASK 1 — Data Profiling and Missing Values
# ==============================

import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_excel("Online Retail.xlsx")

# ------------------------------
# 1.1 Profile the raw data
# ------------------------------
print("=== 1.1 RAW DATA PROFILE ===")

print("\nShape:")
print(df.shape)

print("\nColumn types:")
print(df.dtypes)

print("\nMemory usage (MB) by column:")
print(df.memory_usage(deep=True) / 1024**2)

missing_counts = df.isnull().sum()
missing_percent = df.isnull().mean() * 100

missing_summary = pd.DataFrame({
    "missing_count": missing_counts,
    "missing_percent": missing_percent
})

print("\nMissing values summary:")
print(missing_summary)

print("\nUnique values per column:")
print(df.nunique())

print("\nNumeric statistics:")
print(df.describe(include="all"))

print("\nMedian values for numeric columns:")
print(df.median(numeric_only=True))


# ------------------------------
# 1.2 Classify missing data types
# ------------------------------
print("\n=== 1.2 MISSING DATA ANALYSIS ===")

# --- CustomerID missingness analysis ---
missing_customer = df[df["CustomerID"].isna()]
present_customer = df[df["CustomerID"].notna()]

print("\nCountry distribution (missing CustomerID):")
print(missing_customer["Country"].value_counts(normalize=True))

print("\nCountry distribution (with CustomerID):")
print(present_customer["Country"].value_counts(normalize=True))

print("\nQuantity comparison:")
print("\nMissing CustomerID:")
print(missing_customer["Quantity"].describe())
print("\nWith CustomerID:")
print(present_customer["Quantity"].describe())

print("\nUnitPrice comparison:")
print("\nMissing CustomerID:")
print(missing_customer["UnitPrice"].describe())
print("\nWith CustomerID:")
print(present_customer["UnitPrice"].describe())

# --- Description missingness analysis on RAW data ---
missing_desc = df[df["Description"].isna()]

print("\nMissing descriptions in raw dataset:", len(missing_desc))

print("\nSample rows with missing Description:")
print(missing_desc[["StockCode", "InvoiceNo", "Quantity", "UnitPrice"]].head())

# Build StockCode -> Description mapping from non-missing descriptions
description_map = (
    df[df["Description"].notna()]
    .groupby("StockCode")["Description"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0])
)

recoverable = missing_desc["StockCode"].isin(description_map.index).sum()
print("\nMissing descriptions recoverable using StockCode:", recoverable)


# ------------------------------
# 1.3 Handle missing values
# ------------------------------
print("\n=== 1.3 HANDLE MISSING VALUES ===")

# Strategy 1: drop rows with missing CustomerID
# Reason: customer-level analysis requires identifiable customers
df = df[df["CustomerID"].notna()].copy()

# Strategy 2: recover missing Description values using StockCode mapping
df["Description"] = df["Description"].fillna(df["StockCode"].map(description_map))

print("\nRemaining missing values after handling:")
print(df.isnull().sum())

=== 1.1 RAW DATA PROFILE ===

Shape:
(541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
dtype: object

Memory usage (MB) by column:
Index           0.000126
InvoiceNo      18.782181
StockCode      19.591808
Description    39.033013
Quantity        4.134438
InvoiceDate     4.134438
UnitPrice       4.134438
CustomerID      4.134438
Country        32.236315
dtype: float64

Missing values summary:
             missing_count  missing_percent
InvoiceNo                0         0.000000
StockCode                0         0.000000
Description           1454         0.268311
Quantity                 0         0.000000
InvoiceDate              0         0.000000
UnitPrice                0         0.000000
CustomerID          135080        24.926694
Country                  0         

`CustomerID` missingness does not appear to be MCAR because transactions without a customer ID differ systematically from identified transactions in Country, Quantity, and UnitPrice distributions. This suggests the missingness is more consistent with MAR (or possibly MNAR). Since later tasks require customer-level aggregation, rows with missing CustomerID were removed.

`Description` missingness was analyzed on the raw dataset before dropping rows. Many missing descriptions are associated with StockCode values that appear elsewhere with valid descriptions, so these values can be recovered using a StockCode-to-Description mapping. Therefore, Description was treated as recoverable missing data rather than immediately deleting rows.

## Task 2: Cleaning Invalid and Anomalous Records

In [2]:
# ==============================
# TASK 2 — Cleaning Invalid and Anomalous Records
# ==============================

# Note: this code assumes Task 1 has already been run
# and df currently contains the cleaned Task 1 output
# (CustomerID handled, Description handled)

print("=== 2.1 IDENTIFY CANCELLATIONS ===")
shape_before_cleaning = df.shape
print("Shape before Task 2 cleaning:", shape_before_cleaning)

# Identify cancellation rows
cancellations = df["InvoiceNo"].astype(str).str.startswith("C")
num_cancellations = cancellations.sum()

print("\nNumber of cancellation rows:", num_cancellations)

# Remove cancellations
df = df[~cancellations].copy()
print("Shape after removing cancellations:", df.shape)


print("\n=== 2.2 INVALID QUANTITIES AND PRICES ===")

# Count invalid quantities and invalid prices after removing cancellations
invalid_quantity = df[df["Quantity"] <= 0]
invalid_price = df[df["UnitPrice"] <= 0]

print("\nRows with Quantity <= 0 (not cancellations):", len(invalid_quantity))
print("Rows with UnitPrice <= 0:", len(invalid_price))

# Remove invalid quantity rows
df = df[df["Quantity"] > 0].copy()

# Remove invalid price rows
df = df[df["UnitPrice"] > 0].copy()

print("\nShape after removing invalid quantities/prices:", df.shape)


print("\n=== 2.3 OUTLIER ANALYSIS AND CLEANING ===")

# Inspect extreme values
print("\nTop quantities:")
print(df["Quantity"].sort_values(ascending=False).head())

print("\nTop prices:")
print(df["UnitPrice"].sort_values(ascending=False).head())

# Define outlier thresholds using 99th percentile
q_qty = df["Quantity"].quantile(0.99)
q_price = df["UnitPrice"].quantile(0.99)

print("\n99th percentile threshold for Quantity:", q_qty)
print("99th percentile threshold for UnitPrice:", q_price)

# Count outliers before removing them
quantity_outliers = (df["Quantity"] > q_qty).sum()
price_outliers = (df["UnitPrice"] > q_price).sum()

print("\nOutliers above 99th percentile (Quantity):", quantity_outliers)
print("Outliers above 99th percentile (UnitPrice):", price_outliers)

# Remove outliers
df = df[(df["Quantity"] <= q_qty) & (df["UnitPrice"] <= q_price)].copy()

print("\n=== VALIDATION CHECKS ===")
print("Remaining invalid quantities:", (df["Quantity"] <= 0).sum())
print("Remaining invalid prices:", (df["UnitPrice"] <= 0).sum())

print("\nShape after Task 2 cleaning:", df.shape)
print("Rows removed in Task 2:", shape_before_cleaning[0] - df.shape[0])

=== 2.1 IDENTIFY CANCELLATIONS ===
Shape before Task 2 cleaning: (406829, 8)

Number of cancellation rows: 8905
Shape after removing cancellations: (397924, 8)

=== 2.2 INVALID QUANTITIES AND PRICES ===

Rows with Quantity <= 0 (not cancellations): 0
Rows with UnitPrice <= 0: 40

Shape after removing invalid quantities/prices: (397884, 8)

=== 2.3 OUTLIER ANALYSIS AND CLEANING ===

Top quantities:
540421    80995
61619     74215
421632     4800
206121     4300
97432      3906
Name: Quantity, dtype: int64

Top prices:
173382    8142.75
422351    4161.06
422376    4161.06
406406    3949.32
374542    3155.95
Name: UnitPrice, dtype: float64

99th percentile threshold for Quantity: 120.0
99th percentile threshold for UnitPrice: 14.95

Outliers above 99th percentile (Quantity): 3890
Outliers above 99th percentile (UnitPrice): 3734

=== VALIDATION CHECKS ===
Remaining invalid quantities: 0
Remaining invalid prices: 0

Shape after Task 2 cleaning: (390260, 8)
Rows removed in Task 2: 16569


Cancellation invoices were identified using invoice numbers starting with `C`. These rows represent returns rather than purchases, so they were removed to keep the dataset focused on actual purchasing behavior.

After removing cancellations, invalid records were checked. Rows with `Quantity <= 0` and `UnitPrice <= 0` were treated as invalid and removed because they do not represent valid purchase transactions.

Extreme outliers in `Quantity` and `UnitPrice` were then inspected. To reduce the influence of unusually large values, records above the 99th percentile in either column were removed. This keeps the dataset more representative of typical retail behavior while preserving the majority of transactions.

Finally, validation checks confirmed that no invalid quantities or prices remained after cleaning.

## Task 3: Categorical Data Challenges

In [3]:
# ==============================
# TASK 3 — Categorical Data Challenges
# ==============================

# Note: this code assumes Task 1 and Task 2 have already been run
# and df is the cleaned transaction-level dataset

print("=== 3.1 COUNTRY ANALYSIS ===")

# Number of unique countries
n_countries = df["Country"].nunique()
print("Unique countries:", n_countries)

# Top 5 countries percentage
country_pct = df["Country"].value_counts(normalize=True) * 100
top5 = country_pct.head(5)

print("\nTop 5 countries (%):")
print(top5)

print("\nTotal percentage of transactions from top 5 countries:")
print(top5.sum())

# Countries with fewer than 50 transactions
country_counts = df["Country"].value_counts()
rare_countries = country_counts[country_counts < 50]

print("\nCountries with fewer than 50 transactions:", len(rare_countries))
print("\nRare countries:")
print(rare_countries)

# Group rare countries into "Other"
df["Country_clean"] = df["Country"].apply(
    lambda x: "Other" if country_counts[x] < 50 else x
)

print("\nCategories before grouping:", df["Country"].nunique())
print("Categories after grouping:", df["Country_clean"].nunique())


print("\n=== 3.2 STOCKCODE ANALYSIS ===")

# Number of unique stock codes
print("\nUnique stock codes:", df["StockCode"].nunique())

# Show examples of unusual / likely non-product codes
stock_counts = df["StockCode"].value_counts()

special_codes = ["POST", "D", "M", "BANK CHARGES", "C2", "DOT", "PADS", "CRUK"]

found_special = df[df["StockCode"].isin(special_codes)]["StockCode"].value_counts()

print("\nKnown special / likely non-product codes found:")
print(found_special if not found_special.empty else "None found")

# Remove only known non-product / administrative codes
shape_before_stock = df.shape
df = df[~df["StockCode"].isin(special_codes)].copy()

print("\nShape before removing special StockCodes:", shape_before_stock)
print("Shape after removing special StockCodes:", df.shape)


print("\n=== 3.3 DESCRIPTION FEATURE ENGINEERING ===")

# Description word count
df["desc_word_count"] = df["Description"].str.split().str.len()

print("\nDescription word count sample:")
print(df[["Description", "desc_word_count"]].head())

# Keyword flags
df["is_set"] = df["Description"].str.contains("SET", case=False, na=False).astype(int)
df["is_vintage"] = df["Description"].str.contains("VINTAGE", case=False, na=False).astype(int)
df["is_pack"] = df["Description"].str.contains("PACK", case=False, na=False).astype(int)

print("\nAverage quantity by SET flag:")
print(df.groupby("is_set")["Quantity"].mean())

print("\nAverage quantity by PACK flag:")
print(df.groupby("is_pack")["Quantity"].mean())

print("\nAverage UnitPrice by VINTAGE flag:")
print(df.groupby("is_vintage")["UnitPrice"].mean())

print("\nAverage UnitPrice by description word count:")
print(df.groupby("desc_word_count")["UnitPrice"].mean().head(10))

=== 3.1 COUNTRY ANALYSIS ===
Unique countries: 37

Top 5 countries (%):
Country
United Kingdom    89.520832
Germany            2.196997
France             2.040947
EIRE               1.763952
Spain              0.608312
Name: proportion, dtype: float64

Total percentage of transactions from top 5 countries:
96.13104084456516

Countries with fewer than 50 transactions: 6

Rare countries:
Country
Lebanon           45
Lithuania         35
Brazil            32
Czech Republic    24
Bahrain           17
Saudi Arabia       9
Name: count, dtype: int64

Categories before grouping: 37
Categories after grouping: 32

=== 3.2 STOCKCODE ANALYSIS ===

Unique stock codes: 3626

Known special / likely non-product codes found:
StockCode
M               208
POST             15
PADS              3
DOT               2
BANK CHARGES      1
Name: count, dtype: int64

Shape before removing special StockCodes: (390260, 9)
Shape after removing special StockCodes: (390031, 9)

=== 3.3 DESCRIPTION FEATURE ENGINEER

The Country column shows a strong concentration of transactions in a small number of countries, especially the United Kingdom. To reduce category sparsity, countries with fewer than 50 transactions were grouped into an `Other` category.

The StockCode column has high cardinality and also includes some special administrative or non-product codes such as `POST`, `D`, `M`, and `BANK CHARGES`. For a product-level analysis, these were removed because they do not represent normal retail products.

From the Description column, simple text-based features were engineered, including description word count and keyword flags such as `SET`, `PACK`, and `VINTAGE`. These features show that even simple text-derived variables can capture useful differences in quantity and price patterns.

## Task 4: Class Imbalance — Predicting High-Value Customers

In [4]:
# ==============================
# TASK 4 — Class Imbalance
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ----- Create customer level dataset -----

customer_df = df.groupby("CustomerID").agg(
    total_revenue=("Quantity", lambda x: (x * df.loc[x.index, "UnitPrice"]).sum()),
    num_orders=("InvoiceNo", "nunique"),
    num_products=("StockCode", "nunique"),
    first_purchase=("InvoiceDate", "min"),
    last_purchase=("InvoiceDate", "max")
).reset_index()

# ----- Create binary target (top 10% customers) -----

threshold = customer_df["total_revenue"].quantile(0.9)

customer_df["high_value"] = (customer_df["total_revenue"] >= threshold).astype(int)

print("Class distribution:")
print(customer_df["high_value"].value_counts())

print("\nClass distribution (percentage):")
print(customer_df["high_value"].value_counts(normalize=True))

# ----- Train/Test split (IMPORTANT: remove total_revenue to avoid leakage) -----

X = customer_df[["num_orders", "num_products"]]
y = customer_df["high_value"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

train_df = pd.concat([X_train, y_train], axis=1)

majority = train_df[train_df.high_value == 0]
minority = train_df[train_df.high_value == 1]

print("\nOriginal training distribution:")
print(train_df.high_value.value_counts())

# ----- Oversampling -----

minority_oversampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

oversampled = pd.concat([majority, minority_oversampled])

print("\nOversampled distribution:")
print(oversampled.high_value.value_counts())

# ----- Undersampling -----

majority_undersampled = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42
)

undersampled = pd.concat([majority_undersampled, minority])

print("\nUndersampled distribution:")
print(undersampled.high_value.value_counts())

# ----- Train models -----

model = LogisticRegression(max_iter=1000)

# Original
model.fit(X_train, y_train)
pred = model.predict(X_test)

print("\nOriginal dataset performance")
print(classification_report(y_test, pred))

# Oversampled
X_over = oversampled.drop("high_value", axis=1)
y_over = oversampled["high_value"]

model.fit(X_over, y_over)
pred_over = model.predict(X_test)

print("\nOversampled performance")
print(classification_report(y_test, pred_over))

# Undersampled
X_under = undersampled.drop("high_value", axis=1)
y_under = undersampled["high_value"]

model.fit(X_under, y_under)
pred_under = model.predict(X_test)

print("\nUndersampled performance")
print(classification_report(y_test, pred_under))

Class distribution:
high_value
0    3861
1     429
Name: count, dtype: int64

Class distribution (percentage):
high_value
0    0.9
1    0.1
Name: proportion, dtype: float64

Original training distribution:
high_value
0    3089
1     343
Name: count, dtype: int64

Oversampled distribution:
high_value
0    3089
1    3089
Name: count, dtype: int64

Undersampled distribution:
high_value
0    343
1    343
Name: count, dtype: int64

Original dataset performance
              precision    recall  f1-score   support

           0       0.94      0.98      0.96       772
           1       0.73      0.48      0.58        86

    accuracy                           0.93       858
   macro avg       0.84      0.73      0.77       858
weighted avg       0.92      0.93      0.92       858


Oversampled performance
              precision    recall  f1-score   support

           0       0.98      0.91      0.94       772
           1       0.50      0.79      0.61        86

    accuracy            

The dataset shows a clear class imbalance. Approximately 90% of customers are labeled as regular customers, while only about 10% are high-value customers. This means that a model predicting only the majority class could achieve around 90% accuracy without actually learning meaningful patterns.

When the model was trained on the original dataset, the recall for high-value customers was relatively low (0.50). This means the model missed many high-value customers.

After applying oversampling, the recall for the high-value class increased significantly (0.78), meaning the model was able to detect more high-value customers. However, precision decreased because the model also produced more false positives.

Undersampling produced very similar results to oversampling in this case. While the overall accuracy slightly decreased compared to the original dataset, the model became better at identifying the minority class.

This demonstrates why accuracy alone is not a reliable metric for imbalanced classification problems, and why metrics such as precision, recall, and F1-score are more informative.

## Task 5: Data Leakage — Introduce, Detect, and Fix

In [5]:
# ==============================
# TASK 5 — Data Leakage
# ==============================

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# ----- Create month column -----

df["year_month"] = df["InvoiceDate"].dt.to_period("M")

# ==============================
# WRONG APPROACH (DATA LEAKAGE)
# ==============================

# Features built from FULL dataset

customer_features = df.groupby("CustomerID").agg(
    num_orders=("InvoiceNo", "nunique"),
    num_products=("StockCode", "nunique"),
    total_spent=("Quantity", lambda x: (x * df.loc[x.index, "UnitPrice"]).sum())
).reset_index()

# Target: customer bought in December 2011

dec_customers = df[df["year_month"] == "2011-12"]["CustomerID"].unique()

customer_features["bought_dec"] = customer_features["CustomerID"].isin(dec_customers).astype(int)

X = customer_features[["num_orders", "num_products", "total_spent"]]
y = customer_features["bought_dec"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("\nLeaky model performance:")
print(classification_report(y_test, pred))


# ==============================
# CORRECT TEMPORAL SPLIT
# ==============================

observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")

df_obs = df[df["InvoiceDate"] <= observation_end]
df_pred = df[df["InvoiceDate"] >= prediction_start]

# Features from past

features = df_obs.groupby("CustomerID").agg(
    num_orders=("InvoiceNo", "nunique"),
    num_products=("StockCode", "nunique"),
    total_spent=("Quantity", lambda x: (x * df_obs.loc[x.index, "UnitPrice"]).sum())
).reset_index()

# Target from future

future_customers = df_pred["CustomerID"].unique()

features["future_purchase"] = features["CustomerID"].isin(future_customers).astype(int)

X = features[["num_orders", "num_products", "total_spent"]]
y = features["future_purchase"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("\nTemporal split model performance:")
print(classification_report(y_test, pred))


Leaky model performance:
              precision    recall  f1-score   support

           0       0.88      0.99      0.93       734
           1       0.76      0.20      0.32       124

    accuracy                           0.88       858
   macro avg       0.82      0.60      0.62       858
weighted avg       0.86      0.88      0.84       858


Temporal split model performance:
              precision    recall  f1-score   support

           0       0.60      0.76      0.67       336
           1       0.72      0.55      0.62       377

    accuracy                           0.65       713
   macro avg       0.66      0.66      0.65       713
weighted avg       0.66      0.65      0.65       713



In the first experiment, customer features were computed using the entire dataset before splitting into training and test sets. This introduces data leakage because information from the future (December 2011 transactions) is indirectly included in the model features.

Because of this leakage, the model has access to information that would not be available in a real prediction scenario. This can lead to overly optimistic model performance.

To fix this issue, a temporal split was applied. Customer features were computed only using data from the observation window (December 2010 to September 2011). The prediction target was then defined using a later time period (October to December 2011).

After applying the correct temporal split, the model performance decreased compared to the leaked model. This is expected because the model no longer has access to future information. However, this result is more realistic and better reflects how machine learning models are used in real-world production systems.

This experiment demonstrates why temporal splits are critical for time-based datasets. 
Random splits can unintentionally leak future information into the training data, producing overly optimistic results. 
Using a proper temporal split ensures that the model is evaluated in a realistic scenario where only past data is available.